In [18]:
from pathlib import Path
import heapq
import itertools
import math
import time

import numpy as np
import pandas as pd
from scipy.sparse.csgraph import dijkstra

OUTPUT_DIR = Path(
    r"C:\Users\dalia\OneDrive\Desktop\Enhanced Dijkstra"
)

FACILITIES_PATH = OUTPUT_DIR / "facilities_flagged.csv"
NODES_PATH = OUTPUT_DIR / "Upper_Egypt_nodes.parquet"
EDGES_PATH = OUTPUT_DIR / "Upper_Egypt_edges_weighted.parquet"

print("Output directory exists:", OUTPUT_DIR.exists())
print("Facilities file exists:", FACILITIES_PATH.exists())
print("Nodes file exists:", NODES_PATH.exists())
print("Edges file exists:", EDGES_PATH.exists())

Output directory exists: True
Facilities file exists: True
Nodes file exists: True
Edges file exists: True


In [17]:
nodes = pd.read_parquet(NODES_PATH)
edges = pd.read_parquet(EDGES_PATH)
facilities = pd.read_csv(FACILITIES_PATH)

print("Main graph nodes:", len(nodes))
print("Road edge records:", len(edges))
print("Snapped facilities:", len(facilities))

print("\nNode columns:", nodes.columns.tolist())
print("Edge columns:", edges.columns.tolist())
print("Facility columns:", facilities.columns.tolist())

display(facilities.head())

Main graph nodes: 2300821
Road edge records: 4746637
Snapped facilities: 798

Node columns: ['id', 'lon', 'lat', 'geometry']
Edge columns: ['id', 'u', 'v', 'length', 'oneway', 'maxspeed', 'highway', 'speed_kmh', 'speed_source', 'travel_time_s']
Facility columns: ['canonical_entity_id', 'canonical_name', 'governorate_assigned', 'district', 'address', 'phone_clean', 'phone_type', 'phones_other', 'phone_e164', 'latitude_final', 'longitude_final', 'facility_class', 'birth_service_tier', 'alternate_names', 'facility_category', 'birth_service_evidence', 'phone', 'phones_normalized_all', 'source_files_all', 'source_files', 'source_count', 'n_source_records_total', 'n_same_coord', 'coord_reliable']


,canonical_entity_id,canonical_name,governorate_assigned,district,address,phone_clean,phone_type,phones_other,phone_e164,latitude_final,...,facility_category,birth_service_evidence,phone,phones_normalized_all,source_files_all,source_files,source_count,n_source_records_total,n_same_coord,coord_reliable
0,CE00001,المستشفى الجامعى,Beni Suef,Beni Suef,"Beni Suef, Beni Suef",NaN,NaN,NaN,NaN,29.066100,...,university_general_hospital,general_hospital_type_only_unverified,NaN,NaN,univ,univ,1,1,2,True
1,CE00005,مستشفى وسط البلد الجامعي (المعبر) | مستشفى الح...,Qena,Qena,قنا ـ وسط البلد ـ منطقة معبر الشباب ـ مستشفى ق...,963345726 | 963237405,landline,NaN,2.096335e+10,26.155100,...,university_general_hospital,general_hospital_type_only_unverified,096-3345726,963345726,univ,univ,1,1,1,True
2,CE00007,مستشفيات جامعة أسوان,Aswan,Aswan,"Aswan, Aswan",973480901,landline,0974661403 | 0972430767 | 0973482767 | 0973480...,2.097348e+10,24.086471,...,university_general_hospital,general_hospital_type_only_unverified,097-3480901,973480901,univ,univ,1,1,1,True
3,CE00009,مستشفى بني سويف الجامعي,Beni Suef,Beni Suef,"Beni Suef, Beni Suef",NaN,NaN,NaN,NaN,29.066100,...,university_general_hospital,general_hospital_type_only_unverified,NaN,NaN,univ,univ,1,1,2,True
4,CE00013,المستشفى الجامعي - جامعة الأقصر (قيد الإنشاء),Luxor,New Thebes,"New Thebes, Luxor",NaN,NaN,NaN,NaN,25.687200,...,university_general_hospital,general_hospital_type_only_unverified,NaN,NaN,univ,univ,1,1,1,True


In [5]:
from scipy.sparse import csr_matrix

nodes = (
    nodes.drop_duplicates("id")
         .sort_values("id")
         .reset_index(drop=True)
)

node_ids = nodes["id"].to_numpy(dtype=np.int64)

u_ids = edges["u"].to_numpy(dtype=np.int64)
v_ids = edges["v"].to_numpy(dtype=np.int64)

u_idx = np.searchsorted(node_ids, u_ids)
v_idx = np.searchsorted(node_ids, v_ids)

assert np.all(u_idx < len(node_ids))
assert np.all(v_idx < len(node_ids))
assert np.array_equal(node_ids[u_idx], u_ids)
assert np.array_equal(node_ids[v_idx], v_ids)

N = len(node_ids)

G_time_m = csr_matrix(
    (
        edges["travel_time_s"].to_numpy(dtype=float),
        (u_idx, v_idx)
    ),
    shape=(N, N)
)

G_len_m = csr_matrix(
    (
        edges["length"].to_numpy(dtype=float),
        (u_idx, v_idx)
    ),
    shape=(N, N)
)

G_time_m.sort_indices()
G_len_m.sort_indices()

assert G_time_m.shape == G_len_m.shape
assert G_time_m.shape == (len(nodes), len(nodes))
assert np.all(G_time_m.data >= 0)

print("Routing graph ready:", G_time_m.shape)
print("Directed road connections:", G_time_m.nnz)

Routing graph ready: (2300821, 2300821)
Directed road connections: 4746637


In [6]:

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Preserve the exact node order from the main graph
node_ids = nodes["id"].to_numpy(dtype=np.int64)
node_lon = nodes["lon"].to_numpy(dtype=float)
node_lat = nodes["lat"].to_numpy(dtype=float)

assert nodes["id"].is_unique, "Duplicate node IDs found"
assert np.isfinite(node_lon).all() and np.isfinite(node_lat).all()

# Convert longitude/latitude to 3D unit-sphere coordinates
def to_xyz(lon, lat):
    lon = np.radians(np.asarray(lon, dtype=float))
    lat = np.radians(np.asarray(lat, dtype=float))
    return np.column_stack((
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ))

# Build the spatial index once
node_tree = cKDTree(to_xyz(node_lon, node_lat))

facilities = facilities.copy()

# Prepare required columns
facilities["node_idx"] = -1
facilities["node_id"] = pd.Series(
    [None] * len(facilities), index=facilities.index, dtype="object"
)
facilities["snap_m"] = np.nan
facilities["snap_far"] = True

# Use only facilities marked as having reliable coordinates
reliable = facilities["coord_reliable"].astype(str).str.lower().isin(
    ["true", "1", "yes"]
)

lon = pd.to_numeric(
    facilities["longitude_final"], errors="coerce"
)
lat = pd.to_numeric(
    facilities["latitude_final"], errors="coerce"
)

valid = (
    reliable
    & lon.between(-180, 180)
    & lat.between(-90, 90)
    & lon.notna()
    & lat.notna()
)

valid_idx = facilities.index[valid]

if len(valid_idx) > 0:
    query_xyz = to_xyz(
        lon.loc[valid_idx].to_numpy(),
        lat.loc[valid_idx].to_numpy()
    )

    chord_dist, nearest_idx = node_tree.query(query_xyz, k=1)

    # Convert chord distance to approximate surface distance in meters
    earth_radius_m = 6_371_000
    snap_m = (
        2 * earth_radius_m
        * np.arcsin(np.clip(chord_dist / 2, 0, 1))
    )

    facilities.loc[valid_idx, "node_idx"] = nearest_idx
    facilities.loc[valid_idx, "node_id"] = node_ids[nearest_idx]
    facilities.loc[valid_idx, "snap_m"] = snap_m

    # Flag snaps farther than 1 km for review
    facilities.loc[valid_idx, "snap_far"] = snap_m > 1000

facilities["node_idx"] = facilities["node_idx"].astype(int)

print("Total facilities:", len(facilities))
print("Facilities with reliable coordinates:", int(valid.sum()))
print("Facilities snapped to the road graph:",
      int((facilities["node_idx"] >= 0).sum()))
print("Facilities farther than 1 km:", int(facilities["snap_far"].sum()))

display(facilities[
    ["canonical_name", "node_idx", "node_id", "snap_m", "snap_far"]
].head())


Total facilities: 798
Facilities with reliable coordinates: 736
Facilities snapped to the road graph: 736
Facilities farther than 1 km: 70


,canonical_name,node_idx,node_id,snap_m,snap_far
0,المستشفى الجامعى,341475,5713471676,14.404903,False
1,مستشفى وسط البلد الجامعي (المعبر) | مستشفى الح...,100666,1826888738,7.837984,False
2,مستشفيات جامعة أسوان,351747,5960038478,64.537243,False
3,مستشفى بني سويف الجامعي,341475,5713471676,14.404903,False
4,المستشفى الجامعي - جامعة الأقصر (قيد الإنشاء),2045409,7711046711,104.019611,False


In [7]:
required = [
    "canonical_name",
    "latitude_final",
    "longitude_final",
    "node_idx",
    "node_id",
    "snap_m",
    "snap_far"
]

missing = [col for col in required if col not in facilities.columns]
if missing:
    raise ValueError(f"Missing facility columns: {missing}")

# Keep facilities with valid snapped nodes and reasonable snap distance.
facilities["node_idx"] = pd.to_numeric(
    facilities["node_idx"], errors="coerce"
)

facilities["snap_m"] = pd.to_numeric(
    facilities["snap_m"], errors="coerce"
)

facilities = facilities.dropna(
    subset=[
        "canonical_name",
        "latitude_final",
        "longitude_final",
        "node_idx",
        "snap_m"
    ]
).copy()

facilities["node_idx"] = facilities["node_idx"].astype(int)

facilities = facilities[
    facilities["node_idx"].between(0, len(nodes) - 1)
].copy()

# Exclude facilities flagged as snapped too far from a road.
facilities["snap_far"] = (
    facilities["snap_far"].astype(str).str.lower().isin(["true", "1", "yes"])
)

facilities = facilities[~facilities["snap_far"]].copy()

if facilities.empty:
    raise ValueError("No eligible facilities remain after filtering.")

print("Eligible facilities:", len(facilities))
print("Unique road nodes with hospitals:", facilities["node_idx"].nunique())

display(
    facilities[
        ["canonical_name", "node_idx", "node_id", "snap_m"]
    ].head()
)

Eligible facilities: 728
Unique road nodes with hospitals: 666


,canonical_name,node_idx,node_id,snap_m
0,المستشفى الجامعى,341475,5713471676,14.404903
1,مستشفى وسط البلد الجامعي (المعبر) | مستشفى الح...,100666,1826888738,7.837984
2,مستشفيات جامعة أسوان,351747,5960038478,64.537243
3,مستشفى بني سويف الجامعي,341475,5713471676,14.404903
4,المستشفى الجامعي - جامعة الأقصر (قيد الإنشاء),2045409,7711046711,104.019611


In [8]:
hospital_id_col = (
    "canonical_entity_id"
    if "canonical_entity_id" in facilities.columns
    else "canonical_name"
)

facilities[hospital_id_col] = facilities[hospital_id_col].astype(str)

node_to_hospitals = {}

for _, row in facilities.iterrows():
    node_idx = int(row["node_idx"])

    node_to_hospitals.setdefault(node_idx, []).append({
        "hospital_id": str(row[hospital_id_col]),
        "name": str(row["canonical_name"]),
        "node_idx": node_idx
    })

# Deterministic tie-breaking for hospitals at the same road node.
for node_idx in node_to_hospitals:
    node_to_hospitals[node_idx].sort(
        key=lambda item: (item["name"], item["hospital_id"])
    )

print("Hospital candidate nodes:", len(node_to_hospitals))

Hospital candidate nodes: 666


In [36]:

def enhanced_dijkstra(graph, source_idx, node_to_hospitals):
    """Find the nearest hospital using early-stopping Dijkstra."""

    n = graph.shape[0]

    if not 0 <= source_idx < n:
        raise ValueError("Invalid source node index.")

    if not node_to_hospitals:
        raise ValueError("No candidate hospital nodes available.")

    distances = {source_idx: 0.0}
    previous = {source_idx: None}

    counter = itertools.count()
    queue = [(0.0, next(counter), source_idx)]

    settled_nodes = 0
    best_time = math.inf
    candidates = []

    while queue:
        current_time, _, u = heapq.heappop(queue)

        if current_time > distances.get(u, math.inf):
            continue

        # Stop after all nodes tied at the best hospital time.
        if current_time > best_time:
            break

        settled_nodes += 1

        if u in node_to_hospitals:
            hospital = node_to_hospitals[u][0]

            candidates.append((
                current_time,
                hospital["name"],
                hospital["hospital_id"],
                u
            ))

            best_time = min(best_time, current_time)

        start = graph.indptr[u]
        end = graph.indptr[u + 1]

        for pos in range(start, end):
            v = int(graph.indices[pos])
            weight = float(graph.data[pos])

            if not math.isfinite(weight) or weight < 0:
                raise ValueError(
                    "Edge weights must be finite and non-negative."
                )

            new_time = current_time + weight

            if new_time < distances.get(v, math.inf):
                distances[v] = new_time
                previous[v] = u

                heapq.heappush(
                    queue,
                    (new_time, next(counter), v)
                )

    if not candidates:
        raise ValueError("No reachable hospital found.")

    # Match the baseline's deterministic tie-breaking.
    candidates.sort(key=lambda x: (x[0], x[1], x[2]))

    best_time, name, hospital_id, target = candidates[0]

    path = []
    current = target

    while current is not None:
        path.append(current)
        current = previous[current]

    path.reverse()

    return {
        "hospital_id": hospital_id,
        "hospital_name": name,
        "hospital_node_idx": target,
        "travel_time_s": best_time,
        "travel_time_min": best_time / 60.0,
        "path_idx": path,
        "path_node_ids": nodes.iloc[path]["id"].tolist(),
        "settled_nodes": settled_nodes
    }


In [37]:
from scipy.spatial import cKDTree

def to_xyz(lon, lat):
    lon = np.radians(np.asarray(lon, dtype=float))
    lat = np.radians(np.asarray(lat, dtype=float))

    return np.column_stack((
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ))

node_tree = cKDTree(
    to_xyz(nodes["lon"].to_numpy(), nodes["lat"].to_numpy())
)

def snap_patient_to_node(latitude, longitude):
    _, node_idx = node_tree.query(
        to_xyz([longitude], [latitude]),
        k=1
    )

    return int(node_idx[0])

In [38]:
def baseline_dijkstra(graph, source_idx, node_to_hospitals):
    n = graph.shape[0]

    distances = {source_idx: 0.0}
    previous = {source_idx: None}

    counter = itertools.count()
    queue = [(0.0, next(counter), source_idx)]
    settled_nodes = 0

    while queue:
        current_time, _, u = heapq.heappop(queue)

        if current_time > distances.get(u, math.inf):
            continue

        settled_nodes += 1

        start = graph.indptr[u]
        end = graph.indptr[u + 1]

        for pos in range(start, end):
            v = int(graph.indices[pos])
            weight = float(graph.data[pos])

            if weight < 0:
                raise ValueError("Negative edge weights are not allowed.")

            new_time = current_time + weight

            if new_time < distances.get(v, math.inf):
                distances[v] = new_time
                previous[v] = u

                heapq.heappush(
                    queue,
                    (new_time, next(counter), v)
                )

    candidates = []

    for node_idx, hospitals_at_node in node_to_hospitals.items():
        if node_idx in distances:
            hospital = hospitals_at_node[0]

            candidates.append((
                distances[node_idx],
                hospital["name"],
                hospital["hospital_id"],
                node_idx
            ))

    if not candidates:
        raise ValueError("No reachable hospital found.")

    candidates.sort(key=lambda x: (x[0], x[1], x[2]))

    best_time, name, hospital_id, target = candidates[0]

    path = []
    current = target

    while current is not None:
        path.append(current)
        current = previous[current]

    path.reverse()

    return {
        "hospital_id": hospital_id,
        "hospital_name": name,
        "hospital_node_idx": target,
        "travel_time_s": best_time,
        "travel_time_min": best_time / 60.0,
        "path_idx": path,
        "path_node_ids": nodes.iloc[path]["id"].tolist(),
        "settled_nodes": settled_nodes
    }

In [39]:
rng = np.random.default_rng(42)

hospital_nodes = np.array(list(node_to_hospitals.keys()), dtype=int)
all_nodes = np.arange(len(nodes))

test_sources = np.setdiff1d(all_nodes, hospital_nodes)

if len(test_sources) == 0:
    raise ValueError("No non-hospital node is available for the test.")

source_idx = int(rng.choice(test_sources))

# Baseline
start = time.perf_counter()
baseline_result = baseline_dijkstra(
    G_time_m, source_idx, node_to_hospitals
)
baseline_runtime_ms = (time.perf_counter() - start) * 1000

# Enhanced
start = time.perf_counter()
enhanced_result = enhanced_dijkstra(
    G_time_m, source_idx, node_to_hospitals
)
enhanced_runtime_ms = (time.perf_counter() - start) * 1000

print("Source node index:", source_idx)

print("\nBASELINE")
print("Hospital:", baseline_result["hospital_name"])
print("Travel time (min):", round(baseline_result["travel_time_min"], 2))
print("Settled nodes:", baseline_result["settled_nodes"])
print("Runtime (ms):", round(baseline_runtime_ms, 3))

print("\nENHANCED")
print("Hospital:", enhanced_result["hospital_name"])
print("Travel time (min):", round(enhanced_result["travel_time_min"], 2))
print("Settled nodes:", enhanced_result["settled_nodes"])
print("Runtime (ms):", round(enhanced_runtime_ms, 3))

assert math.isclose(
    baseline_result["travel_time_s"],
    enhanced_result["travel_time_s"],
    rel_tol=1e-9,
    abs_tol=1e-6
)

print("\nPASS: Both algorithms found the same minimum travel time.")

Source node index: 205400

BASELINE
Hospital: الشفاء
Travel time (min): 0.66
Settled nodes: 2157051
Runtime (ms): 8279.73

ENHANCED
Hospital: الشفاء
Travel time (min): 0.66
Settled nodes: 119
Runtime (ms): 2.831

PASS: Both algorithms found the same minimum travel time.


In [41]:
patient_lat = float(input("Patient latitude: "))
patient_lon = float(input("Patient longitude: "))

if not (-90 <= patient_lat <= 90 and -180 <= patient_lon <= 180):
    raise ValueError("Invalid latitude or longitude.")

# Reuse the spatial tree built when snapping facilities
patient_xyz = to_xyz([patient_lon], [patient_lat])[0]
chord_dist, source_idx = node_tree.query(patient_xyz, k=1)

source_idx = int(source_idx)

# Approximate distance from patient to nearest road node
snap_distance_m = (
    2 * 6_371_000
    * np.arcsin(np.clip(chord_dist / 2, 0, 1))
)

print(f"Distance to nearest road node: {snap_distance_m:.1f} m")

if snap_distance_m > 1000:
    print("Warning: Patient is over 1 km from the nearest road node.")

baseline_result = baseline_dijkstra(
    G_time_m, source_idx, node_to_hospitals
)

enhanced_result = enhanced_dijkstra(
    G_time_m, source_idx, node_to_hospitals
)

print("\nRecommended hospital:", enhanced_result["hospital_name"])
print(
    "Estimated travel time:",
    round(enhanced_result["travel_time_min"], 2),
    "minutes"
)
print("Road-node path:", enhanced_result["path_node_ids"])
print("Baseline settled nodes:", baseline_result["settled_nodes"])
print("Enhanced settled nodes:", enhanced_result["settled_nodes"])

assert math.isclose(
    baseline_result["travel_time_s"],
    enhanced_result["travel_time_s"],
    rel_tol=1e-9,
    abs_tol=1e-6
)

Distance to nearest road node: 35.1 m

Recommended hospital: مستشفي دروه
Estimated travel time: 4.85 minutes
Road-node path: [7660452555, 7652913485, 7652913491, 7652913488, 7652913490, 5799382148, 7652913492, 7660455017, 7660455018, 7660455015, 7660455008, 7660455009, 7660455011, 7660455006, 6574520967, 6574520968, 7661035285, 6574520969, 6574520961, 7661035286, 7660452583, 7660452578, 7660452579, 7660452575, 7660452580, 7660452581, 7660452570, 7660452562, 7660452561, 7660452558, 7212082255, 7184233090, 7608558446, 6302475341, 6023189777, 7608558449, 6302475342, 7067040607, 7227141403, 1655797677, 7067040604, 7067040606, 7067040605, 7184233091, 1655797781, 10923031049, 7181740654, 1655797689, 6302570956, 6302466342, 7191418196, 9026339714, 7187759521, 7203206886, 1655797666, 7184235693, 1655797667, 7643838720, 6302466361, 7643838714, 6302466329, 7187759532, 7187759527, 1655797720, 7643838716, 1655797712, 7643838713, 6622606636, 1655797741, 1655797775, 7643838717, 1655797809, 764383871

In [42]:
rng = np.random.default_rng(42)

test_sources = rng.choice(
    len(nodes),
    size=min(30, len(nodes)),
    replace=False
)

evaluation_rows = []

for source_idx in test_sources:
    t0 = time.perf_counter()
    
    baseline = baseline_dijkstra(
        G_time_m, int(source_idx), node_to_hospitals
    )
    baseline_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    enhanced = enhanced_dijkstra(
        G_time_m, int(source_idx), node_to_hospitals
    )
    enhanced_ms = (time.perf_counter() - t0) * 1000

    same_cost = math.isclose(
        baseline["travel_time_s"],
        enhanced["travel_time_s"],
        rel_tol=1e-9,
        abs_tol=1e-6
    )

    evaluation_rows.append({
        "source_idx": int(source_idx),
        "baseline_hospital": baseline["hospital_name"],
        "enhanced_hospital": enhanced["hospital_name"],
        "baseline_time_min": baseline["travel_time_min"],
        "enhanced_time_min": enhanced["travel_time_min"],
        "baseline_settled": baseline["settled_nodes"],
        "enhanced_settled": enhanced["settled_nodes"],
        "baseline_runtime_ms": baseline_ms,
        "enhanced_runtime_ms": enhanced_ms,
        "same_optimal_cost": same_cost
    })

evaluation_df = pd.DataFrame(evaluation_rows)

display(evaluation_df)

print("All costs match:", evaluation_df["same_optimal_cost"].all())
print(
    "Average baseline settled nodes:",
    round(evaluation_df["baseline_settled"].mean(), 2)
)
print(
    "Average enhanced settled nodes:",
    round(evaluation_df["enhanced_settled"].mean(), 2)
)
print(
    "Average baseline runtime (ms):",
    round(evaluation_df["baseline_runtime_ms"].mean(), 3)
)
print(
    "Average enhanced runtime (ms):",
    round(evaluation_df["enhanced_runtime_ms"].mean(), 3)
)

,source_idx,baseline_hospital,enhanced_hospital,baseline_time_min,enhanced_time_min,baseline_settled,enhanced_settled,baseline_runtime_ms,enhanced_runtime_ms,same_optimal_cost
0,1481415,Kom Ombo General Hospital,Kom Ombo General Hospital,16.020717,16.020717,78668,13605,308.7990,48.0611,True
1,1211323,مستشفي دار السلام المركزي سوهاج,مستشفي دار السلام المركزي سوهاج,7.800635,7.800635,42438,3358,155.4246,16.6761,True
2,996279,Qeft Teaching Hospital,Qeft Teaching Hospital,7.814172,7.814172,2157051,6693,7419.9688,21.2885,True
3,1975459,مستشفي الرحمة,مستشفي الرحمة,1.691751,1.691751,2157051,2866,6018.7379,7.9353,True
4,205347,مركز صنبو المتميز,مركز صنبو المتميز,13.479144,13.479144,2157051,11452,5993.1936,30.0363,True
5,216683,دار ابو السعود الطبى بالشرفا,دار ابو السعود الطبى بالشرفا,19.853788,19.853788,2157051,18339,5915.8335,45.6297,True
6,420012,مركز الكوثر الطبي (متعاقد مع التأمين الصحي),مركز الكوثر الطبي (متعاقد مع التأمين الصحي),2.938733,2.938733,2157051,1462,6177.1449,5.4801,True
7,1893025,مستشفى قروى الميمون,مستشفى قروى الميمون,10.124595,10.124595,2157051,10291,6197.1804,30.5993,True
8,1808582,مستشفى 🏥,مستشفى 🏥,8.435348,8.435348,2157051,9492,6856.7079,23.6881,True
9,1604503,مستشفي شارونه المركزي,مستشفي شارونه المركزي,17.279662,17.279662,2157051,3656,6492.2015,13.5416,True


All costs match: True
Average baseline settled nodes: 2017284.47
Average enhanced settled nodes: 6370.2
Average baseline runtime (ms): 5563.805
Average enhanced runtime (ms): 17.708
